In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação por variáveis + Park (comparação) com RF-Temp duplo.
Agora varrendo faixas:
  - 35–50 kHz
  - 35–75 kHz
  - 35–100 kHz
  - 35–125 kHz
Avaliando 46 °C  e 70 °C contra referência 20 °C. E depois mudando a referência
"""



In [ ]:
import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

In [ ]:
REF_TEMP      = 20
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"
TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 46, 70}
SMOOTH_WIN          = 5 #remove ruído residual
TAU_MAX_FRAC        = 0.025 #muda o centroide até bater com a curva de referência (pode mascarar o dano, por isso valor baixo)
ANCHOR_TO_REF_ENDS  = True #força as curvas a coincidirem nas extremidades.
CAP_GAIN_FRAC   = 0.60 #Limite de ganho máximo permitido (±X%) nesse caso 60%
CAP_OFFSET_FRAC = 0.60 #Limite do deslocamento vertical máximo (offset)
CAP_TILT_FRAC   = 0.40 #Limite da correção de inclinação (tilt)
PARK_MAX_SHIFT_FRAC = 0.25 #próximos 3 são parâmetros para controlar o park e ter uma análise rigída do modelo.
PARK_OVERLAP_MIN    = 0.60
PARK_SMOOTH_WIN     = 5

In [ ]:
#Funções para extrair as frequências e tirar um pouco do ruído

def extract_freq_hz(col): #Essa função extrai o valor numérico da frequência (em Hz) do nome da coluna
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None
    
#Filtra todas as colunas de frequência que estão dentro de uma faixa em kHz    
def get_freq_columns(df, fmin_khz, fmax_khz): 
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]


#Reproduz comportamento médio real, eliminando flutuações aleatórias (ou seja ruídos)
def moving_average(arr, win):
    if win<=1 or win%2==0: return arr
    r=win//2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s/float(win)


In [ ]:
def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def spectral_entropy(x):
    ps = np.abs(x)**2
    ps = ps/(np.sum(ps)+1e-12)
    return float(-np.sum(ps*np.log(ps+1e-12)))

    #Calcula a entropia espectral, que mede o grau de dispersão de energia da curva.
    #Quanto mais energia concentrada num único pico, menor a entropia.

def roughness(x):
    return float(np.mean(np.abs(np.diff(x,2))))

    #Identificar curvas com ruído excessivo.

def peak_ratio(x):
    idx = np.argpartition(x, -2)[-2:]
    vals = np.sort(x[idx])
    if len(vals)<2 or vals[1]==0: return 0.0
    return float(vals[1]/(vals[0]+1e-12))

    #Diferenciar modos de vibração distintos pelos picos

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w  = xm*xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f*w, f))
    return num/den

    #É o parâmetro-chave para alinhar curvas

def slope_over_band(f, x):
    return float((x[-1]-x[0])/(f[-1]-f[0] + 1e-12))

    #Usada na etapa de correção de tilt, calcula a inclinação.

In [ ]:

#Função para computar as features, ou seja:
''' Converte o espectro em um vetor que representa a curva:

forma (skew, kurt),

posição (centroid, peak_pos_rel),

intensidade (amp, std),

distribuição (E_bands, entropy),

textura (roughness).'''

def compute_features(X, f):
    X = np.asarray(X, float); n, m = X.shape
    out=[]
    cuts = np.linspace(f[0], f[-1], 6)
    for i in range(n):
        x = X[i]
        mean  = float(np.mean(x))
        std   = float(np.std(x))
        amp   = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        pk_i  = int(np.argmax(x)); peak_pos_rel = pk_i / max(1,(m-1))
        centroid = energy_weighted_centroid(f, x)
        z = (x - mean)/(std + 1e-12)
        skew = float(np.mean(z**3))
        kurt = float(np.mean(z**4))
        E_bands=[]
        for j in range(len(cuts)-1):
            mask = (f>=cuts[j]) & (f<cuts[j+1])
            if mask.sum()<2: E_bands.append(0.0)
            else: E_bands.append(float(np.trapezoid((x[mask]**2), f[mask])))
        ent  = spectral_entropy(x)
        rough= roughness(x)
        pr   = peak_ratio(x)
        out.append([mean,std,amp,slope,peak_pos_rel,centroid,skew,kurt,*E_bands,ent,rough,pr])
    cols = ["mean","std","amp","slope","peak_pos_rel","centroid","skew","kurt",
            "E_b1","E_b2","E_b3","E_b4","E_b5","entropy","roughness","peak_ratio"]
    return np.array(out, float), cols

In [ ]:
def fit_feature_vs_temp_models(F, T, names):
    models = {}
    T = np.asarray(T, float).reshape(-1,1)
    for j, name in enumerate(names):
        lr = LinearRegression().fit(T, F[:,j])
        models[name] = lr
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(lr.predict(Tref)[0]) for name,lr in models.items()}


In [ ]:
def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]; amp_t  = targets["amp"]
    slope_t= targets["slope"]; centroid_t = targets.get("centroid", None)
    mean_x = float(x.mean()); amp_x  = float(x.max() - x.min())
    slope_x= slope_over_band(f, x)

    # 1. offset (nível médio)
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset

    # 2. ganho (amplitude)
    gain = 1.0 if amp_x<=1e-9 else float(amp_t/amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain*(x - mean_t)

    # 3. tilt (inclinação)
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1]-f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal

    # 4. centróide (deslocamento espectral)
    if centroid_t is not None:
        cent_x = energy_weighted_centroid(f, x)
        delta_c = centroid_t - cent_x
        tau_max = TAU_MAX_FRAC * (f[-1]-f[0])
        tau = float(np.clip(delta_c, -tau_max, tau_max))
        if abs(tau) > 1e-12:
            x = shift_interp(x, f, tau)

    # 5. ancoragem nas extremidades
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0] - y_ref[0]; e1 = x[-1] - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr
    return x